In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
from astropy.table import Table
from astropy.io import ascii
from scipy.spatial import Delaunay
import pandas as pd
from itertools import combinations
import networkx as nx
import ast

In [ ]:
filt_n1 = Table.read("/content/drive/MyDrive/DESI/data/LRG_NGC_data_filt1.ecsv").to_pandas()
filt_n2 = Table.read("/content/drive/MyDrive/DESI/data/LRG_NGC_data_filt2.ecsv").to_pandas()

In [ ]:
n_random = 10
class_ngc1 = {}
for i in range(n_random):
    table = Table.read(
        f"/content/drive/MyDrive/DESI/classification/LRG_NGC_1_random_{i}_filt.fits"
    ).to_pandas()

    col = table[f'class_{i}']
    if col.dtype == object and isinstance(col.iloc[0], (bytes, bytearray)):
        col = col.apply(lambda x: x.decode('utf-8') if isinstance(x, (bytes, bytearray)) else x)

    class_ngc1[i] = dict(zip(table['TARGETID'], col))


class_ngc2 = {}
for i in range(n_random):
    table = Table.read(
        f"/content/drive/MyDrive/DESI/classification/LRG_NGC_2_random_{i}_filt.fits").to_pandas()

    col = table[f'class_{i}']
    if col.dtype == object and isinstance(col.iloc[0], (bytes, bytearray)):
        decoded_col = col.apply(lambda x: x.decode('utf-8') if isinstance(x, (bytes, bytearray)) else x)
    else:
        decoded_col = col
    class_ngc2[i] = dict(zip(table['TARGETID'], decoded_col))

In [ ]:
connections = {}
for j in range(1):
        file = f"/content/drive/MyDrive/DESI/connections/LRG_NGC_1_random_{j}_neighbors.parquet"
        df = pd.read_parquet(file)
        connections[j] = df

In [ ]:
n_random = 1
for i in range(n_random):
    connections[i][f'class_{i}'] = connections[i]['TARGETID'].map(class_ngc1[i])

In [ ]:
def func_connections(df, classification, type_data, i, j):
    id_to_type = dict(zip(df['TARGETID'], df['type']))
    id_to_class = dict(zip(df['TARGETID'], df[f'class_{j}']))
    mask_data_class = (df['type'] == type_data) & (df[f'class_{j}'] == classification)
    df_data_class = df[mask_data_class].copy()

    data_class_connections = {}

    for _, row in df_data_class.iterrows():
        center_id = row['TARGETID']
        neighbors_list = row.get(f'neighbor_ids_{type_data}', [])

        if not isinstance(neighbors_list, (list, tuple, np.ndarray)):
            continue

        valid_neighbors = [
            int(n) for n in neighbors_list
            if id_to_type.get(int(n)) == type_data and id_to_class.get(int(n)) == classification
        ]

        if valid_neighbors:
            data_class_connections[center_id] = valid_neighbors
    rows = [
        (center, neighbor)
        for center, neighbors in data_class_connections.items()
        for neighbor in neighbors
    ]
    df_connections = pd.DataFrame(rows, columns=["TARGETID", f"TARGETID_{classification}"])

    filename = f"/content/drive/MyDrive/DESI/connections_groups/LRG_NGC_{i}_random_{j}_neighbors_{classification}.parquet"
    df_connections.to_parquet(filename, index=False)
    print(f"Guardado: {filename} ({len(df_connections)} filas)")

    return df_connections

In [ ]:
%%time
for i in range(1):
      func_connections(connections[i],'void',type_data='rand',i=1,j=i)

Guardado: /content/drive/MyDrive/DESI/connections_groups/LRG_NGC_1_random_0_neighbors_void.parquet (12496 filas)
CPU times: user 2.81 s, sys: 134 ms, total: 2.95 s
Wall time: 3.03 s


In [ ]:
%%time
for i in range(1):
      func_connections(connections[i],'filament',type_data='data',i=1,j=i)

Guardado: /content/drive/MyDrive/DESI/connections_groups/LRG_NGC_1_random_0_neighbors_filament.parquet (2896442 filas)
CPU times: user 25 s, sys: 444 ms, total: 25.5 s
Wall time: 27.2 s


In [ ]:
%%time
for i in range(1):
      func_connections(connections[i],'knot',type_data='data',i=1,j=i)

Guardado: /content/drive/MyDrive/DESI/connections_groups/LRG_NGC_1_random_0_neighbors_knot.parquet (12718 filas)
CPU times: user 2.78 s, sys: 83.3 ms, total: 2.86 s
Wall time: 3.61 s
